# ProphetGP 퀵스타트

학습 → 다음 실험 추천 → 지정 조건 예측 → CSV 병합까지 한 바퀴 돌려 봅니다.

**실행 전**
- 프로젝트 루트에서 `pip install -e ".[dev]"` 로 설치해 두세요.
- 셀은 위에서부터 순서대로 실행하세요 (`Run All` 권장).
- 이 노트북은 `notebooks/` 또는 저장소 루트에서 열어도 됩니다.

**이 예시에서 쓰는 파일**
- 설정: `configs/example_open_reactants.yaml`
- 데이터: `data/sample/emission_experiments_demo.csv`


In [ ]:
# 필요하면 주석을 해제해 설치합니다.
# %pip install -e ".[dev]"


In [ ]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# notebooks/에서 실행해도 프로젝트 루트를 찾습니다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "example_open_reactants.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "emission_experiments_demo.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "sample" / "new_batch_example.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config:", CONFIG_PATH.name)
print("Data:", DATA_PATH.name)


## 1) 사용 가능한 featuriser

설정에서 고를 수 있는 분자 featuriser 목록입니다. 이 예시는 `topo_physchem`을 씁니다.


In [ ]:
available_featurisers = pipeline.featurizers.available()
print("count:", len(available_featurisers))
print(available_featurisers)


## 2) 학습

CSV를 읽어 GP를 학습합니다. 이후 셀은 여기서 만든 `artifacts`를 사용합니다.


In [ ]:
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])


## 3) 다음 실험 후보 추천

- `best_output`: 목표에 가깝거나 성능이 좋아 보이는 쪽
- `best_information`: 모델이 덜 아는 쪽(탐색)

출력에서 자주 보는 필드:
- `mapped_reactants_input`, `Temperature` — 실험에 쓸 조건
- `predicted_target_mean` / `predicted_target_std` — 예측값과 불확실성
- `total_score` — 순위에 사용된 점수


In [ ]:
n_candidates = 5
strategy = "best_output"  # 또는 "best_information"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(f"\n- candidate_{idx}")
    print("  mapped reactants:", row.get("mapped_reactants_input"))
    print("  Temperature:", row.get("Temperature"))
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])
    print("  total_score:", row["total_score"])


## 4) 지정한 조건에 대한 예측

추천과 달리, 직접 고른 반응물·조건의 예측값만 조회합니다.


In [ ]:
query_inputs = [
    {"reactants": "2,6-Diaminonaphthalene", "Temperature": 250.0},
    {"reactants": "5-amino-1,10-phenanthroline|salicylic acid", "Temperature": 250.0},
]
prediction_result = pipeline.predict_targets(artifacts, query_inputs)

for idx, row in enumerate(prediction_result.predictions, 1):
    print(f"- query_{idx}")
    print("  reactants:", row["reactants_input"])
    print("  conditions:", row["conditions"])
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])


## 5) 신규 배치 병합 (선택)

`new_batch_example.csv`를 기존 데이터에 이어 붙입니다. 결과는 `data/raw/` 아래에 저장됩니다.


In [ ]:
if not NEW_DATA_PATH.exists():
    print("Skip append: missing", NEW_DATA_PATH)
else:
    MERGED_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
    print("Merged rows:", len(merged))
    print("Saved to:", MERGED_OUT_PATH)
